# LongiHealth

## Reproducible Longitudinal EHR Analytics Pipeline

**Author:** [Sepideh Moafi]

**LongiHealth** is a reproducible research software pipeline for longitudinal EHR analytics and exploratory hospital mortality prediction, built on the **MIMIC-IV Clinical Database Demo v2.2**.

## Table of Contents

1. Project Overview
2. Scientific Rationale
3. Dataset
4. Pipeline Architecture
5. Reproducibility Guarantees
6. Environment Setup
7. Cohort Construction
8. Laboratory Feature Engineering
9. Vital-Sign Feature Engineering
10. Final Clinical Feature Matrix
11. Patient-Level Splitting
12. Leakage-Safe Preprocessing
13. Baseline Models
14. Evaluation & Results
15. Limitations
16. Reproducibility & Testing
17. Citation

## 1. Project Overview

LongiHealth implements an **end-to-end, leakage-safe machine learning pipeline** for exploratory hospital mortality prediction from ICU admission data.

### Design goals

- **Reproducibility:** deterministic and configurable
- **Leakage safety:** no outcome variables enter the feature space
- **Modularity:** isolated modules in `src/longihealth/`
- **Testability:** automated tests verify structural and leakage guarantees
- **Portability:** Dockerfile and GitHub Actions CI
- **Honesty:** limitations stated explicitly

### What this project is NOT

- Not a clinical decision support system
- Not validated on a real-world population
- Does not claim clinical performance

## 2. Scientific Rationale

### Why ICU mortality?

Hospital mortality prediction from ICU admission is a classic benchmark for clinical machine learning: clear binary outcome, strong clinical relevance, well-established predictor families.

### Why leakage-safety?

In EHR machine learning, data leakage is the single most common source of over-optimistic results. LongiHealth explicitly prevents leakage by excluding outcome and post-outcome variables, fitting preprocessing on training data only, and splitting at the patient level.

### Why MIMIC-IV Demo?

The **MIMIC-IV Clinical Database Demo v2.2** is publicly accessible, fully de-identified, and structurally identical to the full MIMIC-IV. It is ideal for methodological demonstration, while its small size (100 patients) makes it unsuitable for clinical conclusion.

## 3. Dataset

**Source:** MIMIC-IV Clinical Database Demo v2.2

**URL:** https://physionet.org/content/mimic-iv-demo/2.2/

**DOI:** 10.13026/dp1f-ex47

### Cohort summary

| Quantity | Value |
|---|---:|
| Patients | 100 |
| Hospital admissions | 128 |
| Positive outcomes | 15 |
| Overall mortality rate | 11.7% |

> MIMIC data are not redistributed with this repository. Users must obtain them through PhysioNet and comply with the applicable data-use terms.

## 4. Pipeline Architecture

```
Raw MIMIC-IV Demo
       |
       v
Data ingestion  (src/longihealth/data.py)
       |
       v
Cohort construction  (src/longihealth/cohort.py)
       |
       v
24-hour feature window from ICU intime
       |
   +---+---------------+
   v                   v
Lab features      Vital features
   |                   |
   +--------+----------+
            v
Final clinical matrix  (features.py)
            |
            v
Patient-level split  (split.py)
            |
            v
Leakage-safe preprocessing  (preprocess.py)
            |
            v
Baseline models  (models.py)
            |
            v
Evaluation  (evaluate.py)
            |
            v
Automated tests + Docker + CI
```

## 5. Reproducibility Guarantees

| Guarantee | Implementation |
|---|---|
| Deterministic random state | `random_seed: 42` |
| Configurable parameters | `configs/default.yaml` |
| Modular pipeline | each stage in `src/longihealth/` |
| Leakage-safe preprocessing | fit on training split only |
| Patient-level splitting | with overlap assertions |
| Automated tests | `pytest tests/` |
| Containerization | `Dockerfile` |
| Continuous integration | `.github/workflows/ci.yml` |
| License | MIT |
| Citation metadata | `CITATION.cff` |

## 6. Environment Setup

```bash
git clone https://github.com/<user>/longihealth.git
cd longihealth
pip install -r requirements.txt
python -m longihealth.pipeline
pytest -q tests
```

Or via Docker:

```bash
docker build -t longihealth .
docker run --rm longihealth
```

## 7. Cohort Construction

### Index event

The prediction index is the first ICU stay associated with each hospital admission.

### Feature window

All features are extracted from `[index_date, index_date + 24 hours)` where `index_date = ICU intime`.

### Outcome

`hospital_expire_flag` — used only as the target, never as a feature.

In [ ]:
import sys
sys.path.insert(0, "src")

from longihealth.config import load_config
from longihealth.cohort import build_cohort, cohort_summary

cfg = load_config("configs/default.yaml")
cohort = build_cohort(cfg)

summary = cohort_summary(cohort)
for k, v in summary.items():
    print(f"{k:22s}: {v}")

## 8. Laboratory Feature Engineering

For each `(admission, itemid)` pair, six statistics are computed: count, mean, min, max, std, abnormal_count.

Result: **336 laboratory features** after prevalence filtering.

In [ ]:
from longihealth.data import load_dictionary
from longihealth.features_lab import extract_lab_features, build_lab_matrix

lab_features = extract_lab_features(cfg, cohort)
d_labitems = load_dictionary(cfg, "d_labitems")

lab_matrix = build_lab_matrix(
    lab_features, cohort,
    min_observed_fraction=cfg["features"]["lab"]["min_observed_fraction"],
    dictionary=d_labitems,
)

lab_cols = [c for c in lab_matrix.columns if c.startswith(("lab_", "abnormal_count_"))]
print(f"Lab features: {len(lab_cols)}")
print(f"Matrix shape: {lab_matrix.shape}")

## 9. Vital-Sign Feature Engineering

Only a whitelisted set of ICU item IDs is used (10 items). For each `(admission, itemid)`, five statistics are computed: count, mean, min, max, std.

Result: **50 vital-sign features** (10 items x 5 stats).

In [ ]:
from longihealth.features_vital import extract_vital_features, build_vital_matrix

vital_features = extract_vital_features(cfg, cohort)
d_items = load_dictionary(cfg, "d_items")

vital_matrix = build_vital_matrix(vital_features, cohort, d_items)
vital_cols = [c for c in vital_matrix.columns if c.startswith("vital_")]
print(f"Vital features: {len(vital_cols)}")
print(f"Matrix shape: {vital_matrix.shape}")

## 10. Final Clinical Feature Matrix

Laboratory, vital-sign, and cohort metadata are merged into a single admission-level matrix.

### Leakage exclusion list

The following columns are never used as features: identifiers, timestamps, administrative variables, demographics, outcome, and post-outcome variables.

In [ ]:
from longihealth.features import merge_clinical_features, feature_summary

clinical = merge_clinical_features(cohort, lab_matrix, vital_matrix)

s = feature_summary(clinical)
for k, v in s.items():
    print(f"{k:22s}: {v}")

## 11. Patient-Level Splitting

Splitting is done at the patient level, not admission level. If the same patient appears in both training and test, the model can memorize patient-specific patterns.

| Split | Patients | Admissions | Positive outcomes |
|---|---:|---:|---:|
| Train | 70 | 89 | 11 |
| Validation | 15 | 20 | 2 |
| Test | 15 | 19 | 2 |

In [ ]:
from longihealth.split import split_patient_level, split_summary

train, val, test = split_patient_level(clinical, cfg)
ss = split_summary(train, val, test)
for split_name, stats in ss.items():
    print(f"{split_name:12s}: {stats}")

## 12. Leakage-Safe Preprocessing

All preprocessing is fit on the training split only.

1. Feature filtering (train only)
2. Median imputation (fit on train)
3. Standardization (fit on train)

This prevents validation/test information from leaking into model development.

In [ ]:
from longihealth.preprocess import preprocess_splits, save_preprocessed

prep = preprocess_splits(train, val, test, cfg)
print(f"Features kept:    {len(prep['features'])}")
print(f"Features removed: {len(prep['removed'])}")
print(f"X_train shape:    {prep['X_train'].shape}")
print(f"X_val shape:      {prep['X_val'].shape}")
print(f"X_test shape:     {prep['X_test'].shape}")

## 13. Baseline Models

Three baseline classifiers are trained:

| Model | Class imbalance handling |
|---|---|
| Logistic Regression | `class_weight='balanced'` |
| Random Forest | `class_weight='balanced'` |
| Gradient Boosting | `sample_weight=balanced` |

Hyperparameters are defined in `configs/default.yaml`.

In [ ]:
from longihealth.models import (
    build_logistic_regression,
    build_random_forest,
    build_gradient_boosting,
    fit_model,
)

X_train, X_val, X_test = prep["X_train"], prep["X_val"], prep["X_test"]
y_train, y_val, y_test = prep["y_train"], prep["y_val"], prep["y_test"]

predictions = {}

for name, builder in [
    ("Logistic Regression", build_logistic_regression),
    ("Random Forest", build_random_forest),
    ("Gradient Boosting", build_gradient_boosting),
]:
    model = builder(cfg)
    fit_model(model, X_train, y_train)
    predictions[name] = {
        "validation": {"y_true": y_val, "proba": model.predict_proba(X_val)[:, 1]},
        "test": {"y_true": y_test, "proba": model.predict_proba(X_test)[:, 1]},
    }
    print(f"{name} trained")

## 14. Evaluation & Results

Metrics: AUROC, AUPRC, F1, Precision, Recall, Balanced Accuracy, Brier.

### Test-set results

| Model | AUROC | AUPRC | Brier |
|---|---:|---:|---:|
| Logistic Regression | 0.235 | 0.103 | 0.435 |
| **Random Forest** | **0.706** | **0.225** | **0.113** |
| Gradient Boosting | 0.676 | 0.208 | 0.179 |

**Random Forest achieved the highest test AUROC.**

### Top features

1. `abnormal_count_creatinine_50912`
2. `lab_min_anion_gap_50868`
3. `lab_min_creatinine_50912`
4. `lab_mean_anion_gap_50868`
5. `lab_mean_creatinine_50912`

Clinically plausible: creatinine (renal failure) and anion gap (metabolic acidosis) are established ICU severity markers.

In [ ]:
from longihealth.evaluate import compare_models, best_model_by_metric
import pandas as pd
pd.set_option("display.max_columns", None)

comparison = compare_models(predictions, threshold=cfg["evaluation"]["threshold"])
display(comparison)

best = best_model_by_metric(comparison, "AUROC", "test")
print(f"Best test model by AUROC: {best}")

### Figures

In [ ]:
from pathlib import Path
from IPython.display import Image, display as ipy_display

fig_dir = Path("artifacts/evaluation/figures")
for name in [
    "test_auroc_comparison.png",
    "test_auprc_comparison.png",
    "test_brier_comparison.png",
    "random_forest_top15_features.png",
]:
    p = fig_dir / name
    if p.exists():
        print(f"\n{name}")
        ipy_display(Image(filename=str(p)))

## 15. Limitations

**This section is essential.** Any honest ML project must state its limitations clearly.

### Data limitations

- Small cohort: 100 patients, 128 admissions
- Few outcomes: only 15 positive outcomes; the test split has **2 positives**
- Single-center: MIMIC-IV Demo is a subset of a single US hospital
- No external validation

### Statistical limitations

- AUROC on 2 positive outcomes has an extremely wide confidence interval
- F1, precision, recall are zero at threshold 0.5 under extreme class imbalance
- Brier score is dominated by class imbalance

### Interpretation

**These results are exploratory.** They demonstrate a reproducible pipeline, not a clinical model. No clinical claims are made or should be inferred.

### How this would be improved in a real study

1. Use the full MIMIC-IV dataset (credentialed access)
2. Increase patient count by an order of magnitude
3. Use temporal cross-validation
4. Perform external validation on eICU or a second institution
5. Report confidence intervals via bootstrap
6. Calibrate probability outputs
7. Use survival / time-to-event modeling instead of binary classification

## 16. Reproducibility & Testing

**11 tests pass**, covering config loading, cohort summary, leakage prevention, feature counting, patient-level split integrity, preservation of all patients, train-only feature filtering, and imputation correctness.

### CI

`.github/workflows/ci.yml` runs the test suite on every push and pull request.

### Docker

```bash
docker build -t longihealth .
docker run --rm longihealth
```

## 17. Citation

### MIMIC-IV

Johnson AEW, Bulgarelli L, Shen L, et al. MIMIC-IV, a freely accessible electronic health record dataset. Scientific Data. 2023. DOI: 10.1038/s41597-022-01899-x

### MIMIC-IV Demo

DOI: 10.13026/dp1f-ex47

### LongiHealth

See `CITATION.cff` in the repository root.

*This notebook is a reproducible research artifact. Clinical data are not redistributed; users must obtain MIMIC-IV Demo through PhysioNet.*